In [1]:
# =========================================================
# BƯỚC 1: CÀI ĐẶT & ÉP PYTHON NHẬN DIỆN THƯ MỤC
# =========================================================
import os
import sys
import json
from types import ModuleType

%cd /kaggle/working
if not os.path.exists("DocLayout-YOLO"):
    !git clone --depth 1 https://github.com/opendatalab/DocLayout-YOLO.git

%cd DocLayout-YOLO
!pip install -q -e .

# Ép Kernel nhận thư mục
repo_dir = "/kaggle/working/DocLayout-YOLO"
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

%cd /kaggle/working

# =========================================================
# BƯỚC 2: "HACK" BỘ NHỚ ĐỂ FIX LỖI MODULE HUB
# =========================================================
if "doclayout_yolo.utils.callbacks.hub" not in sys.modules:
    dummy_hub = ModuleType("doclayout_yolo.utils.callbacks.hub")
    dummy_hub.callbacks = {}
    sys.modules["doclayout_yolo.utils.callbacks.hub"] = dummy_hub

# =========================================================
# BƯỚC 3: CHẠY INFERENCE QUÉT TOÀN BỘ THƯ MỤC
# =========================================================
from pathlib import Path
import shutil
import cv2
from doclayout_yolo import YOLOv10 

print("✅ Import YOLOv10 thành công!")

# ---------------------------------------------------------
# CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# ---------------------------------------------------------
MODEL_PATH = "/kaggle/input/models/notpitomon/htd-box-doclayoutyolo/pytorch/default/1/DocLayoutYOLO first version.pt"

# THAY ĐỔI LỚN TẠI ĐÂY: Quét toàn bộ thư mục test
TEST_IMG_DIR = Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images")

image_paths = []
if TEST_IMG_DIR.exists():
    # Gom tất cả file có đuôi jpg, jpeg, png
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        image_paths.extend(TEST_IMG_DIR.glob(ext))
else:
    raise FileNotFoundError(f"Không tìm thấy thư mục: {TEST_IMG_DIR}")

# Sắp xếp lại list để chạy có thứ tự (tùy chọn)
image_paths = sorted(image_paths)

# Đổi thư mục output để chứa file JSONL
OUT_DIR = Path("/kaggle/working/inference_output")
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

META_PATH = OUT_DIR / "metadata.jsonl"

IMG_SIZE = 1280
CONF = 0.35  
MAX_DET = 200

assert Path(MODEL_PATH).exists(), f"Không tìm thấy model: {MODEL_PATH}"
print(f"📸 Tìm thấy tổng cộng {len(image_paths)} ảnh cần dự đoán.")

# Khởi tạo model
print("\nĐang load model...")
model = YOLOv10(MODEL_PATH)

# Chạy dự đoán và trích xuất dữ liệu
print("🚀 Bắt đầu trích xuất tọa độ...")
count_images = 0

with open(META_PATH, 'w', encoding='utf-8') as f_out:
    for img_path in image_paths:
        results = model.predict(
            source=str(img_path),  # Cần ép kiểu str() vì Path object đôi khi làm YOLO lú
            imgsz=IMG_SIZE,
            conf=CONF,
            max_det=MAX_DET,
            verbose=False
        )

        r = results[0]
        # Lấy kích thước ảnh gốc (height, width)
        img_h, img_w = r.orig_shape

        regions = []
        for box in r.boxes:
            # YOLO trả về xyxy (x_min, y_min, x_max, y_max)
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            
            # Chuyển sang format (x, y, width, height)
            w = x2 - x1
            h = y2 - y1
            
            # Lấy tên nhãn dạng text
            cls_id = int(box.cls[0])
            cls_name = model.names[cls_id]
            
            regions.append({
                "bbox": [round(x1, 2), round(y1, 2), round(w, 2), round(h, 2)],
                "type": cls_name
            })

        # Xây dựng cục JSON chuẩn theo format của tập train
        data = {
            "file_name": f"images/{img_path.name}",
            "image_width": img_w,
            "image_height": img_h,
            "annotation_source": "yolov10_prediction",
            "regions": regions
        }

        # Ghi vào file
        f_out.write(json.dumps(data, ensure_ascii=False) + "\n")
        count_images += 1
        
        # In tiến độ để tiện theo dõi
        if count_images % 50 == 0:
            print(f"⏳ Đã xử lý {count_images}/{len(image_paths)} ảnh...")

print("\n--- HOÀN TẤT ---")
print(f"Đã xử lý thành công: {count_images} ảnh")
print(f"File metadata đã được lưu tại: {META_PATH}")

/kaggle/working
Cloning into 'DocLayout-YOLO'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 277 (delta 41), reused 237 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (277/277), 10.79 MiB | 35.17 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/kaggle/working/DocLayout-YOLO
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for doclayout_yolo (pyproject.toml) ... done
/kaggle/working
✅ Import YOLOv10 thành công!
📸 Tìm thấy tổng cộng 386 ảnh cần dự đoán.

Đang load model...
🚀 Bắt đầu trích xuất tọa độ...
⏳ Đã xử lý 50/386 ảnh...
⏳ Đã xử lý 100/386 ảnh...
⏳ Đã xử lý 150/386 ảnh...
⏳ Đã xử lý 200/386 ảnh...
⏳ Đã xử lý 250/386 ảnh...
⏳ Đã xử lý 300/386 ảnh...
⏳ Đã xử lý 350/386 ảnh...

--- HOÀN

In [2]:
# # =========================================================
# # BƯỚC 1: CÀI ĐẶT & ÉP PYTHON NHẬN DIỆN THƯ MỤC
# # =========================================================
# import os
# import sys
# from types import ModuleType

# %cd /kaggle/working
# if not os.path.exists("DocLayout-YOLO"):
#     !git clone --depth 1 https://github.com/opendatalab/DocLayout-YOLO.git

# %cd DocLayout-YOLO
# !pip install -q -e .

# # Ép Kernel nhận thư mục
# repo_dir = "/kaggle/working/DocLayout-YOLO"
# if repo_dir not in sys.path:
#     sys.path.insert(0, repo_dir)

# %cd /kaggle/working

# # =========================================================
# # BƯỚC 2: "HACK" BỘ NHỚ ĐỂ FIX LỖI MODULE HUB
# # =========================================================
# if "doclayout_yolo.utils.callbacks.hub" not in sys.modules:
#     dummy_hub = ModuleType("doclayout_yolo.utils.callbacks.hub")
#     dummy_hub.callbacks = {}
#     sys.modules["doclayout_yolo.utils.callbacks.hub"] = dummy_hub

# # =========================================================
# # BƯỚC 3: CHẠY INFERENCE VỚI YOLOv10
# # =========================================================
# from pathlib import Path
# import shutil
# import cv2

# # SỬA Ở ĐÂY: Dùng YOLOv10 thay vì YOLO để load đúng Predictor của v10
# from doclayout_yolo import YOLOv10 

# print("✅ Import YOLOv10 thành công!")

# # ---------------------------------------------------------
# # CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# # ---------------------------------------------------------
# MODEL_PATH = "/kaggle/input/models/notpitomon/htd-box-doclayoutyolo/pytorch/default/1/DocLayoutYOLO first version.pt"

# IMAGE_PATHS = [
#     "/kaggle/input/datasets/quii29/rukopys-dataset/test/images/0414761c-70d3-48f4-ad2d-23b965d67685.jpg",
#     "/kaggle/input/datasets/quii29/rukopys-dataset/test/images/04cf4a71-95e9-442b-8b09-ffbba912e1e9.jpg",
#     "/kaggle/input/datasets/quii29/rukopys-dataset/test/images/0bfb347f-9b53-4c8d-b971-e7ddc2fe4ea9.jpg",
#     "/kaggle/input/datasets/quii29/rukopys-dataset/test/images/1a7d3f8a-9f10-4918-a768-9b5c04caf859.jpg",
#     "/kaggle/input/datasets/quii29/rukopys-dataset/test/images/24f7d274-b6ce-4495-939f-9528a1fe42f4.jpg",
# ]

# OUT_DIR = Path("/kaggle/working/specific_images_boxed")
# if OUT_DIR.exists():
#     shutil.rmtree(OUT_DIR)
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# IMG_SIZE = 1280
# CONF = 0.35  
# MAX_DET = 200
# LINE_WIDTH = 2
# # Đã xóa IOU vì YOLOv10 không xài cái này để triệt tiêu box nữa

# assert Path(MODEL_PATH).exists(), f"Không tìm thấy model: {MODEL_PATH}"
# for p in IMAGE_PATHS:
#     assert Path(p).exists(), f"Không tìm thấy ảnh: {p}"

# # Khởi tạo model bằng YOLOv10
# print("\nĐang load model...")
# model = YOLOv10(MODEL_PATH)

# # Chạy dự đoán
# print("Bắt đầu vẽ box...")
# saved_files = []

# for img_path in IMAGE_PATHS:
#     results = model.predict(
#         source=img_path,
#         imgsz=IMG_SIZE,
#         conf=CONF,
#         max_det=MAX_DET,
#         verbose=False
#     )

#     r = results[0]
    
#     # Vẽ kết quả
#     plotted = r.plot(
#         line_width=LINE_WIDTH,
#         labels=True,
#         conf=False
#     )

#     save_path = OUT_DIR / Path(img_path).name
#     cv2.imwrite(str(save_path), plotted)
#     saved_files.append(save_path)

# print("\n--- HOÀN TẤT ---")
# for f in saved_files:
#     print(f"Đã lưu: {f.name}")
# print(f"Thư mục chứa ảnh: {OUT_DIR}")